<a href="https://colab.research.google.com/github/whyaskidk91-web/saturnGPT/blob/main/saturnGPT_QAchatbot_18N.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**SATURN GPT**

This is a basic Transformer model in GPT2 Style,
cut to 16million parameters due to compute costs(A100).

BUILD:
- uses BPE tokenizer(~2.2k tokens),Adam,flash attention,
bf16,transformer(8 blocks with 6 heads per block),
optional dropout, gradient clipping,cosine learn rate,
warmup
- Pretrain 1.40 · midtrain 2.40 · SFT ~2.16

DATA:
- ~3 phase training(basic english,chat bot conversations,small talk)
- ~1.2billion characters as data
- ~360million tokens in the data
- ~220million words in the data

AGENDA:
- [0] - [5] : necessarities and set up to chat
- [A] - [D] : prepare the data(once)
- [I] - [X] : training

HOW TO RUN:
- This model creates and uses files from your drive to work
- Run [A] - [D],[I] - [X] to create these files, make sure to run [0] - [5] in cell order when executing training or loading data
- A100 suggested for computing tho T4 works too just slower




In [ ]:
#[0] IMPORT

import torch,re,math as m,time,os
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from google.colab import drive
DRIV = '/content/drive/MyDrive/'
drive.mount('/content/drive') #connects to google drive
dev = "cuda" if torch.cuda.is_available() else "cpu"
END = "<|endoftext|>"
pat = re.compile(r" ?\w+| ?[^\w\s]+|\s+")
ctx = 512 #context window
print(dev,ctx)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cpu 512


In [ ]:
#[A] LOAD DATASETS
# -  only do once
!wget -q https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt -O tiny_train.txt
from datasets import load_dataset
stories = open('tiny_train.txt').read(800_000_000)
smol = load_dataset("HuggingFaceTB/smol-smoltalk",split="train")
parts,total = [],0
for r in smol:
  msgs = [x for x in r["messages"] if x["role"] in ("user","assistant")]
  for u,a in zip(msgs[0::2],msgs[1::2]):
    s = "User: " + u["content"] + "\nAssistant: " + a["content"] + END
    parts.append(s);total += len(s)
  if total > 400_000_000: break
chat = "".join(parts)
del parts,smol
print(len(stories) / 1e6,"M chars stories |",len(chat) / 1e6,"M chars chat")

800.0 M chars stories | 400.001932 M chars chat


In [ ]:
#[1] TOKENIZER FUNCTIONS

def get_pairs(ids):
  counts = {}
  for pair in zip(ids,ids[1:]):
    if pair in counts:
      counts[pair] += 1
    else:
      counts[pair] = 1
  return counts

def merge(ids,pair,new_id):
  out = []
  i = 0
  while i < len(ids):
    if (i+1) < len(ids) and (ids[i],ids[i + 1]) == pair:
      out.append(new_id)
      i += 2
    else:
      out.append(ids[i])
      i += 1
  return out

def clean(s): return bad.sub('',s)

cache = {}
def encode_word(w):
  if w in cache: return cache[w] #every word gets encoded only once
  ids = [stoi[c] for c in w]
  while len(ids) > 1:
    pairs = get_pairs(ids)
    best = min(pairs,key = lambda p: merges.get(p, float('inf'))) #merges in order
    if best not in merges: break
    ids = merge(ids,best,merges[best])
  if len(cache) < 3_000_000: cache[w] = ids
  return ids

def encode(text):
  num_text = []
  for w in pat.findall(clean(text)): #splits text into words,punct. and spaces
    num_text += encode_word(w)
  return num_text

def decode(ids):
  return ''.join(vocab[i] for i in ids)


In [ ]:
#[B] BUILD TOKENS
# -  only do once

sample = stories[:20_000_000] + chat[:20_000_000]
cc = Counter(sample)
chars = sorted(c for c in cc if cc[c] > 50) #ignore rare chars
k0 = len(chars)
stoi = {c: a for a,c in enumerate(chars)}
bad = re.compile('[^' + re.escape(''.join(chars)) + ']')

words = { word: count for word,count in Counter(pat.findall(clean(sample))).items()
 if count >= 3} #ignore rare words
split = {word: [stoi[c] for c in word] for word in words}

merges = {}
for n in range(2000):
  pairs = {}
  for word,cnt in words.items():
    ids = split[word]
    for p in zip(ids,ids[1:]):
      pairs[p] = pairs.get(p,0) + cnt
  if pairs == {}: break
  top = max(pairs,key=pairs.get)
  merges[top] = k0 + n
  for word in words:
    ids = split[word]
    if len(ids) > 1 and top[0] in ids: #skip words without the pair in it
      split[word] = merge(ids,top,k0 + n)
  if (n + 1) % 200 == 0: print(n)

torch.save({'characters': chars,
            'merges': merges},
           DRIV + 'tok_18M.pt')


199
399
599
799
999
1199
1399
1599
1799
1999


In [ ]:
#[2] LOAD BUILT TOKENS

loaded = torch.load(DRIV + 'tok_18M.pt',weights_only=False)
chars = loaded['characters'];merges = loaded['merges']
stoi = {c: a for a,c in enumerate(chars)}
itos = {a: c for a,c in enumerate(chars)}
bad = re.compile('[^' + re.escape(''.join(chars)) + ']')
vocab = {i: itos[i] for i in range(len(chars))}
for (a,b),new_id in merges.items():
  vocab[new_id] = vocab[a] + vocab[b]
k = len(chars) + len(merges)
cache = {}
print(k,decode(encode("Hello! How are you today?")))

2132 Hello! How are you today?


In [ ]:
#[C] ENCODE DATA
# -  only do once
def encode_big(text,chunk=10_000_000):
  text_encode = []
  for i in range(0,len(text),chunk): #chunk to save RAM
    e = torch.tensor(encode(text[i:(i+chunk)]),dtype=torch.int16) #4x less memory
    text_encode.append(e)
    print(len(text_encode))
  return torch.cat(text_encode)

big_stories = encode_big(stories)
torch.save(big_stories,DRIV + 'pre_18M.pt')
big_chat = encode_big(chat)
torch.save(big_chat,DRIV + 'mid_18M.pt')
print(len(big_chat)/1e6,"M CHAT TOKEN |",len(big_stories)/1e6,"M STORY TOKEN")

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
130.898151 M CHAT TOKEN | 228.699244 M STORY TOKEN


In [ ]:
#[I] LOAD ENCODED DATA
# -  only do when training
pre_load = torch.load(DRIV + 'pre_18M.pt').to(dev)
mid_load = torch.load(DRIV + 'mid_18M.pt').to(dev)
cut = int(0.99 * len(pre_load))
pre_train,pre_test = pre_load[:cut],pre_load[cut:]
cut = int(0.99 * len(mid_load))
mid_train,mid_test = mid_load[:cut],mid_load[cut:]
print(len(pre_train),len(pre_test),len(mid_train),len(mid_test))

226412251 2286993 129589169 1308982


In [ ]:
#[II] BATCH
# -   only do when training
import torch

def batch(bD,it):
  idx = torch.randint(0,len(bD) - ctx - 1,size=(it,),device=dev)
  X = (bD[idx[:,None] + torch.arange(ctx,device=dev)]).long() #Embedding needs 64bit ints
  Y = (bD[idx[:,None] + torch.arange(1,ctx + 1,device=dev)]).long()
  return X,Y

In [ ]:
#[3][III] TRANSFORMER
# -    only do when training
import torch.nn as nn
import torch.nn.functional as F


d = 384
nh = 6
it = 64
block_num = 8
drp = 0.0

class Head(nn.Module):
  def __init__(self, hs):
    super().__init__()
    self.hs = hs
    self.drop = nn.Dropout(drp)
    #ATTENTION HEAD:
    self.query = nn.Linear(d,hs,bias=False)
    self.key = nn.Linear(d,hs,bias=False)
    self.value = nn.Linear(d,hs,bias=False)

  def forward(self,x):
    B,T,_ = x.shape
    q = self.query(x).unsqueeze(1) #fast attention wants heads dimension
    kk = self.key(x).unsqueeze(1)
    v = self.value(x).unsqueeze(1)
    computed = F.scaled_dot_product_attention(q,kk,v,is_causal=True) #(= q·kᵀ,
    # scale, mask, softmax, ·v in one fused step
    return computed.squeeze(1) #go back to the needed dimensions

class MultiHead(nn.Module):
  def __init__(self):
    super().__init__()
    self.drop = nn.Dropout(drp)
    self.heads = nn.ModuleList([Head(d//nh) for _ in range(nh)])
    self.proj = nn.Linear(d,d)


  def forward(self,x):
    out = torch.cat([h(x) for h in self.heads],dim=-1)
    return self.drop(self.proj(out))

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
    nn.Linear(d,d*4),
    nn.ReLU(),
    nn.Linear(d*4,d),
    nn.Dropout(drp),
    )

  def forward(self,x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self):
    super().__init__()
    self.ln1 = nn.LayerNorm(d)
    self.att = MultiHead()
    self.ln2 = nn.LayerNorm(d)
    self.ff = FeedForward()

  def forward(self,x):
    x = x + self.att(self.ln1(x))
    x = x + self.ff(self.ln2(x))
    return x


class Net(nn.Module):
  def __init__(self):
    super().__init__()
    self.tok = nn.Embedding(k,d)
    self.pos = nn.Embedding(ctx,d)
    self.blocks = nn.Sequential(*[Block() for _ in range(block_num)])
    self.ln = nn.LayerNorm(d)
    self.out = nn.Linear(d,k)


  def forward(self,x):
    B,T = x.shape
    x = self.tok(x) + self.pos(torch.arange(T,device=dev))
    x = self.blocks(x)
    x = self.ln(x)
    return self.out(x)


In [ ]:
#[IV] CHECK BATCH
# -   only do when training
model = Net().to(dev)
opt = torch.optim.Adam(model.parameters(),lr=6e-4)
print(sum(p.numel() for p in model.parameters())/1e6)
print_me,_ = batch(pre_train,it)
print(model(print_me).shape)

16.02338
torch.Size([64, 512, 2132])


In [ ]:
#[V] TRAIN
# -  only do when training
def eval_lm(Dt,n=20):
  model.eval()
  with torch.no_grad():
    total = 0
    for _ in range(n):
      evX,evY = batch(Dt,it)
      total += F.cross_entropy(model(evX).view(-1,k),evY.view(-1)).item()
  model.train()
  return total / n


def train_lm(trainD,testD,T,lr_base,path,start=0):
  model.train()
  loss_percent = 0
  t0 = time.time()
  for i in range(start,T):
    X,Y = batch(trainD,it)
    with torch.autocast(dev,dtype=torch.bfloat16): #bf16 is around 2x faster
      logits = model(X)
      loss = F.cross_entropy(logits.view(-1,k),Y.view(-1))
    loss_percent += loss.item()
    lr = lr_base * 0.5 * (1 + m.cos(m.pi * i / T))
    lr *= min(1,(i+1)/200) #warmup
    if (i + 1) % 250 == 0:
      print("STEP:",(i + 1),"TEST LOSS:",eval_lm(testD),"TRAIN LOSS:",
            loss_percent/250,"TIME:",(time.time() - t0) / 60,"min")
      loss_percent = 0
    if (i + 1) % 1000 == 0 or (i + 1) == T:
      torch.save({'model': model.state_dict(),
                  'opt': opt.state_dict(),
                  'step': (i + 1)
                  },path)
    opt.zero_grad()
    loss.backward()
    for g in opt.param_groups:
      g['lr'] = lr
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0) #cap huge gradients
    opt.step()

In [ ]:
#[_] RESUME WHEN DISCONNECTED
# -  only do when accidentally disconnected and training
ckpt = torch.load(DRIV + 'ckpt_pre_18M.pt')
model.load_state_dict(ckpt['model'])
opt.load_state_dict(ckpt['opt'])
start = ckpt['step']
print("resuming at step", start)
T = 6000
lr_base = 6e-4
path = DRIV + 'ckpt_pre_18M.pt'
train_lm(pre_train,pre_test,T,lr_base,path,start)
torch.save(model.state_dict(),DRIV + 'pre_18M_model.pt')

In [ ]:
#[VI] TRAIN PHASE 1
# -   only do when training
T = 6000
lr_base = 6e-4
path = DRIV + 'ckpt_pre_18M.pt'
train_lm(pre_train,pre_test,T,lr_base,path)
torch.save(model.state_dict(),DRIV + 'pre_18M_model.pt')

STEP: 250 TEST LOSS: 3.463564372062683 TRAIN LOSS: 4.6424595012664795 TIME: 0.4243483463923136 min
STEP: 500 TEST LOSS: 2.694061744213104 TRAIN LOSS: 3.0499031410217285 TIME: 0.8391126394271851 min
STEP: 750 TEST LOSS: 2.235409367084503 TRAIN LOSS: 2.4473713417053222 TIME: 1.2534807721773784 min
STEP: 1000 TEST LOSS: 1.9903850913047791 TRAIN LOSS: 2.111701234340668 TIME: 1.6679178476333618 min
STEP: 1250 TEST LOSS: 1.892356163263321 TRAIN LOSS: 1.933778527736664 TIME: 2.0904958645502725 min
STEP: 1500 TEST LOSS: 1.7827567100524901 TRAIN LOSS: 1.822970051765442 TIME: 2.5048324783643086 min
STEP: 1750 TEST LOSS: 1.7140067100524903 TRAIN LOSS: 1.7468282775878907 TIME: 2.9194286982218425 min
STEP: 2000 TEST LOSS: 1.680219554901123 TRAIN LOSS: 1.689585392475128 TIME: 3.3339915831883746 min
STEP: 2250 TEST LOSS: 1.6198385059833527 TRAIN LOSS: 1.6385968146324157 TIME: 3.7569647868474325 min
STEP: 2500 TEST LOSS: 1.5939413726329803 TRAIN LOSS: 1.5990785727500916 TIME: 4.171412102381388 min
STE

In [ ]:
#[D][VII] LOAD DATA PHASE 3
# -       only do when train and once while loading data
import os
from datasets import load_dataset
chat_train = load_dataset("HuggingFaceTB/everyday-conversations-llama3.1-2k",split="train_sft")
chat_test = load_dataset("HuggingFaceTB/everyday-conversations-llama3.1-2k",split="test_sft")
END = "<|endoftext|>"
path = '/content/drive/MyDrive/chat_data_18M.pt'


def build(split):
  BX,BY,BM = [],[],[]
  for r in split:
    msgs = r["messages"]
    for u,a in zip(msgs[0::2],msgs[1::2]):
      p = encode("User: " + clean(u["content"]) + "\nAssistant:")
      q = encode(" " + clean(a["content"]) + END)
      ids = p + q
      mask = [0] * len(p) + [1] * len(q) #only answer tokens count
      if len(ids) > (ctx + 1): continue
      pad = ctx + 1 - len(ids) #padding
      ids += [0] * pad
      mask += [0] * pad
      BX.append(ids[:-1]);BY.append(ids[1:]);BM.append(mask[1:])
  return (torch.tensor(BX,device=dev),torch.tensor(BY,device=dev),
          torch.tensor(BM,device=dev,dtype=torch.float))

if os.path.exists(path):
  CX,CY,CM,TX,TY,TM = torch.load(path)
else:
  CX,CY,CM = build(chat_train)
  TX,TY,TM = build(chat_test)
  torch.save((CX,CY,CM,TX,TY,TM),path)
print(CX.shape,TX.shape)

README.md:   0%|          | 0.00/3.03k [00:00<?, ?B/s]

data/train_sft-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.05MB            

data/train_sft-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test_sft-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  124kB            

data/test_sft-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train_sft split:   0%|          | 0/2260 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/119 [00:00<?, ? examples/s]

torch.Size([8625, 512]) torch.Size([455, 512])


In [ ]:
#[VIII] FINE TUNE BATCH
# -     only run when training
def chat_batch(cbX,cbY,cbM,it):
  idx = torch.randint(0,len(cbX),(it,),device=dev)
  return cbX[idx],cbY[idx],cbM[idx]

def masked_loss(logits,mY,mM): #loss of only assisstants answers
  l = F.cross_entropy(logits.view(-1,k),mY.view(-1),reduction='none')
  return (l * mM.view(-1)).sum() / mM.sum()

In [ ]:
#[IX] TRAIN PHASE 2
# -   only run when training
model = Net().to(dev)
model.load_state_dict(torch.load('/content/drive/MyDrive/pre_18M_model.pt'))
opt = torch.optim.Adam(model.parameters(),lr=3e-4)
T = 3000
lr_base = 3e-4
path = DRIV + 'ckpt_mid_18M.pt'
train_lm(mid_train,mid_test,T,lr_base,path)
torch.save(model.state_dict(),DRIV + 'mid_18M_model.pt')

STEP: 250 TEST LOSS: 3.344493865966797 TRAIN LOSS: 4.17791623878479 TIME: 0.4139311989148458 min
STEP: 500 TEST LOSS: 2.8996559381484985 TRAIN LOSS: 3.1109409084320068 TIME: 0.828544557094574 min
STEP: 750 TEST LOSS: 2.7549607038497923 TRAIN LOSS: 2.844922101020813 TIME: 1.2431591272354126 min
STEP: 1000 TEST LOSS: 2.6481916427612306 TRAIN LOSS: 2.705938669204712 TIME: 1.657603347301483 min
STEP: 1250 TEST LOSS: 2.5610929131507874 TRAIN LOSS: 2.6158495016098025 TIME: 2.081385040283203 min
STEP: 1500 TEST LOSS: 2.489024293422699 TRAIN LOSS: 2.5466131830215453 TIME: 2.4957778692245483 min
STEP: 1750 TEST LOSS: 2.4797022700309754 TRAIN LOSS: 2.494024366378784 TIME: 2.910299285252889 min
STEP: 2000 TEST LOSS: 2.4387984752655028 TRAIN LOSS: 2.460719807624817 TIME: 3.3245760679244993 min
STEP: 2250 TEST LOSS: 2.395413887500763 TRAIN LOSS: 2.4289558868408205 TIME: 3.7477368831634523 min
STEP: 2500 TEST LOSS: 2.4358551263809205 TRAIN LOSS: 2.4128509845733643 TIME: 4.162109605471293 min
STEP: 2

In [ ]:
#[X] TRAIN PHASE 3
# -  only do when training
import math as m
model = Net().to(dev)
model.load_state_dict(torch.load(DRIV + 'mid_18M_model.pt'))
opt = torch.optim.Adam(model.parameters(),lr=1e-4)
T,lr_base,sft_it = 1000,1e-4,16
for i in range(T):
  model.train()
  X,Y,M = chat_batch(CX,CY,CM,sft_it)
  with torch.autocast(dev,dtype=torch.bfloat16):
    loss = masked_loss(model(X),Y,M)
  lr = lr_base * 0.5 * (1 + m.cos(m.pi * i / T))
  for g in opt.param_groups: g['lr'] = lr
  opt.zero_grad();loss.backward();opt.step()
  if (i + 1) % 250 == 0:
    model.eval()
    with torch.no_grad():
      Xt,Yt,Mt = chat_batch(TX,TY,TM,sft_it*4)
      tl = masked_loss(model(Xt),Yt,Mt).item()
    print(f"step {i + 1} train {loss.item():.3f} test {tl:.3f}")
    torch.save(model.state_dict(),'/content/drive/MyDrive/chat_ckpt_18M.pt')
torch.save(model.state_dict(),'/content/drive/MyDrive/chat_model_18M.pt')


step 250 train 2.160 test 2.248
step 500 train 2.185 test 2.129
step 750 train 1.764 test 1.993
step 1000 train 1.836 test 2.158


In [ ]:
#[4] LOAD FULL MODEL

model = Net().to(dev)
model.load_state_dict(torch.load(DRIV + 'chat_model_18M.pt',map_location=torch.device('cpu')))

<All keys matched successfully>

In [ ]:
#[5] RUN FULL MODEL

model.eval()
while True:
  Temp = 0.5
  msg = input("You: ")
  if msg == "q": break
  ids = encode("User: " + clean(msg) + "\nAssistant:")
  out = []
  with torch.no_grad():
    for _ in range(200):
      x = torch.tensor([ids[-ctx:]],device=dev)
      probs = F.softmax(model(x)[:,-1,:] / Temp,dim=1)[0]
      t = torch.multinomial(probs,1).item()
      ids.append(t);out.append(t)
      if decode(out).endswith(END): break
  print("Bot:",decode(out).replace(END,"").strip())

You: a
Bot: Hello! How can I help you today?
You: q
